# Agents & Tools with Guardrails

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohswedd/vincio/blob/main/examples/notebooks/03_agents_and_tools.ipynb)

[Vincio](https://github.com/Ohswedd/vincio) turns plain Python functions into **governed agent actions**. Every tool call is permission-checked, schema-validated, and (for writes) approval-gated *before* it runs.

This notebook builds a small refund agent two ways:

- **(A)** the one-line `tool_agent(...)` front door — write tools are denied / left pending by default.
- **(B)** the verbose `ContextApp` path — `add_tool(...)` with explicit RBAC permissions and `approval_required=True`.
- **(C)** notes on RBAC / ABAC permissions and bounded loops.

Everything runs **fully offline** on the deterministic mock provider — no API keys, no network.

In [ ]:
!pip install -q vincio

## Setup: the provider helper

`ContextApp` needs a provider. We default to the bundled deterministic **mock** provider so the notebook runs offline, end to end.

> **To run against a real model, set `VINCIO_PROVIDER` and the matching API key** (e.g. `os.environ["VINCIO_PROVIDER"]="openai"`, `os.environ["VINCIO_MODEL"]="gpt-4o-mini"`, `os.environ["OPENAI_API_KEY"]="sk-..."`). The code below switches automatically.

The optional `script=` argument only feeds the *offline* mock a fixed sequence of tool calls so the guardrail behaviour is reproducible. A real model decides its own tool calls and ignores it.

In [ ]:
import os
import warnings

# Experimental front-door helpers emit a friendly warning; quiet it for the tour.
warnings.simplefilter("ignore")

from vincio.providers import MockProvider, build_provider


def _provider(script=None):
    """Offline mock by default; a real provider when VINCIO_PROVIDER is set."""
    name = os.environ.get("VINCIO_PROVIDER", "mock")
    if name == "mock":
        # `script` only scripts the offline mock; real providers decide for themselves.
        return MockProvider(script=script), "mock-1"
    return build_provider(name), os.environ.get("VINCIO_MODEL", "gpt-4o-mini")

## Define two tools: a read and a write

A plain Python function becomes a typed tool — Vincio derives its input schema from the type hints. We define:

- `lookup_order` — a **read** tool (safe, no special scope).
- `issue_refund` — a **write** tool (irreversible; must be gated).

In [ ]:
def lookup_order(order_id: str) -> dict:
    """Read tool: look up an order by id (no special scope needed)."""
    return {"order_id": order_id, "amount": 49.0, "status": "delivered", "age_days": 5}


def issue_refund(order_id: str, amount: float) -> dict:
    """Write tool: issue a refund (irreversible side effect)."""
    return {"refunded": amount, "order_id": order_id}

## (A) The one-line front door: `tool_agent(...)`

`tool_agent(tools=[...], writes=[...])` wires a governed model+tool loop in a single call:

- read tools run freely,
- **write tools are denied by default** and surfaced as *pending approvals*.

So a one-shot reply can never silently fire a write. We script the offline mock to *attempt* both tools; watch the refund get held back.

In [ ]:
from vincio.tasks import tool_agent

# Offline script: the mock "decides" to look up the order, then attempt the refund.
script = [
    {"tool_call": {"name": "lookup_order", "arguments": {"order_id": "A-1001"}}},
    {"tool_call": {"name": "issue_refund", "arguments": {"order_id": "A-1001", "amount": 49.0}}},
    "I looked up order A-1001; the refund is pending human approval before it runs.",
]

provider, model = _provider(script=script)
agent = tool_agent(
    tools=[lookup_order],     # read tools: allowed
    writes=[issue_refund],    # write tools: gated, pending by default
    provider=provider,
    model=model,
    name="refund-agent",
)

result = agent.run("Look up order A-1001 and issue a $49 refund.")

print("final answer :", result.output)
print("tool verdicts:", [(r.tool_name, r.status) for r in result.tool_results])
print("approvals    :", [(r.tool, r.status) for r in agent.approvals])
print("pending      :", [r.tool for r in agent.pending_approvals])

### Approving a write

`agent.approve("issue_refund")` records a standing decision, so a later run may execute that write. This is the human-in-the-loop seam: approve, then re-run.

In [ ]:
approve_script = [
    {"tool_call": {"name": "issue_refund", "arguments": {"order_id": "A-1001", "amount": 49.0}}},
    "Refund issued for order A-1001.",
]
provider, model = _provider(script=approve_script)

approved_agent = tool_agent(
    tools=[lookup_order],
    writes=[issue_refund],
    provider=provider,
    model=model,
).approve("issue_refund")  # pre-approve the write tool by name

result = approved_agent.run("Issue the approved $49 refund for order A-1001.")
print("tool verdicts:", [(r.tool_name, r.status) for r in result.tool_results])
print("approvals    :", [(r.tool, r.status) for r in approved_agent.approvals])

## (B) The verbose `ContextApp` path

The front door is sugar over `ContextApp`. Here we wire it by hand to see the full contract:

- `add_tool(read, side_effects="read")` — no scope required.
- `add_tool(write, permissions=["refunds:write"], side_effects="write", approval_required=True)` — needs an RBAC scope **and** approval.

First, inspect the auto-derived schema and the declared contract.

In [ ]:
from vincio import ContextApp

provider, model = _provider()  # contract inspection needs no model calls
app = ContextApp(name="refunds", provider=provider, model=model)

app.add_tool(lookup_order, side_effects="read")
app.add_tool(
    issue_refund,
    permissions=["refunds:write"],   # RBAC: caller must hold this scope
    side_effects="write",
    approval_required=True,           # ...and a human must approve
)

spec = app.tool_registry.get("issue_refund").spec
print("issue_refund schema fields :", sorted(spec.input_schema.get("properties", {})))
print("issue_refund requires scope:", spec.permissions, "| approval:", spec.approval_required)
print("lookup_order requires scope:", app.tool_registry.get("lookup_order").spec.permissions)

### Running a ReAct agent — the write is blocked

The default runtime principal holds **no** `refunds:write` scope, so the permission check blocks the side-effecting call *before* it runs and records it with status `denied`. The agent still reasons to a conclusion; the irreversible action never fires.

Every action also rides the platform's hash-chained audit log, so the denial is verifiable after the fact.

In [ ]:
run_script = [
    {"tool_call": {"name": "lookup_order", "arguments": {"order_id": "A-1001"}}},
    {"tool_call": {"name": "issue_refund", "arguments": {"order_id": "A-1001", "amount": 49.0}}},
    "Order A-1001 reviewed; refund withheld pending an authorized, approved caller.",
]
provider, model = _provider(script=run_script)
app = ContextApp(name="refunds", provider=provider, model=model)
app.add_tool(lookup_order, side_effects="read")
app.add_tool(
    issue_refund,
    permissions=["refunds:write"],
    side_effects="write",
    approval_required=True,
)

agent = app.agent(planner="react", max_steps=6)   # bounded: at most 6 steps
state = agent.run("Look up order A-1001 and issue a $49 refund.")

print("answer        :", state.final_answer)
print("tool verdicts :", [(r.tool_name, r.status) for r in state.tool_results])
print("audit intact  :", app.audit.verify_chain())

## (C) RBAC / ABAC permissions and bounded loops

**Permissions are the gate.** A tool declares the scopes it requires via `permissions=[...]`. The runtime checks them against the calling principal's scopes *before* execution:

- **RBAC** (role-based): a principal carries scopes like `refunds:write`; a tool that requires a scope the principal lacks is **denied** (what you saw in B).
- **ABAC** (attribute-based): policies can additionally condition on attributes of the request/resource (amount thresholds, region, ownership) — same deny-before-execute seam, richer predicate.
- **Approval** (`approval_required=True`) is orthogonal to scopes: even an authorized caller routes a write through the approval surface — the front door's `pending_approvals` / `approve(...)` in (A).

**Loops are bounded.** Every agent runs under an explicit budget — `max_steps` caps the think→act→observe loop so an agent can never spin indefinitely. Combined with permission checks and approvals, a run is guaranteed to terminate and to never fire an unauthorized write.

The cell below recaps the governing contract for each tool — the single source of truth the runtime enforces on every call.

In [ ]:
for name in ("lookup_order", "issue_refund"):
    s = app.tool_registry.get(name).spec
    scopes = str(s.permissions or [])
    print(
        f"{name:14} side_effects={s.side_effects:5} "
        f"scopes={scopes:16} approval_required={s.approval_required}"
    )

print()
print("Bound loop budget for the ReAct agent above: max_steps=6")
print("=> permission-checked, schema-validated, approval-gated, and guaranteed to terminate.")